# Order Isomorphism: Divisors of 120 and Down-Sets of Join-Irreducibles

This notebook demonstrates the categorical and order-theoretic isomorphism between:
1. $\mathcal{D}(120)$: The lattice of divisors of $120 = 2^3 \cdot 3 \cdot 5$, partially ordered by divisibility ($a \le b \iff a \mid b$).
2. $\mathcal{O}(S)$: The lattice of divisor-closed subsets (down-sets / order ideals) of the poset of join-irreducible elements $S = \{2, 3, 4, 5, 8\}$, partially ordered by set inclusion ($U_1 \le U_2 \iff U_1 \subseteq U_2$).

### Theoretical Background: Birkhoff's Representation Theorem
By **Birkhoff's Representation Theorem for Finite Distributive Lattices**, any finite distributive lattice $L$ is naturally order-isomorphic to the lattice of down-sets $\mathcal{O}(J(L))$ of its poset of join-irreducible elements $J(L)$:
$$
L \cong \mathcal{O}(J(L))
$$
For $L = \mathcal{D}(120)$:
- The join-irreducible elements are prime powers $p^k > 1$ dividing $120$:
  $$J(\mathcal{D}(120)) = \{2^1, 2^2, 2^3, 3^1, 5^1\} = \{2, 4, 8, 3, 5\} = S$$
- The canonical order-preserving bijection $\Phi: \mathcal{D}(120) \xrightarrow{\cong} \mathcal{O}(S)$ is:
  $$\Phi(d) = \{s \in S : s \mid d\}$$
- The inverse map is:
  $$\Psi(U) = \operatorname{lcm}(U \cup \{1\}) = \prod_{p \in \{2, 3, 5\}} \max(\{1\} \cup \{s \in U : s \text{ is a power of } p\})$$

In [ ]:
from itertools import combinations
from sage.all import *
from sage.combinat.posets.posets import Poset
from dzack_research.preamble.categories.sets.owned_sets import Sets, placement_of

print('Owned Poset Category:', Sets().PartiallyOrdered().Finite())

## 1. Construct $\mathcal{D}(120)$ (Divisors Poset)

In [ ]:
# Divisors of 120 ordered by divisibility
divs_120 = [d for d in Integer(120).divisors()]
P_div = Poset((divs_120, lambda a, b: b % a == 0))

print(f'Divisors count: {P_div.cardinality()}')
print(f'Divisors: {divs_120}')
print(f'Category placement: {placement_of(P_div)}')

## 2. Construct $\mathcal{O}(S)$ (Divisor-Closed Subsets of $S = \{2, 3, 4, 5, 8\}$)

In [ ]:
S = [2, 3, 4, 5, 8]

def is_divisor_closed(U):
    """Return True if U is a down-set in S with respect to divisibility."""
    for x in U:
        for y in S:
            if x % y == 0 and y not in U:
                return False
    return True

all_subsets = [frozenset(sub) for size in range(len(S) + 1) for sub in combinations(S, size)]
closed_subsets = [sub for sub in all_subsets if is_divisor_closed(sub)]

P_closed = Poset((closed_subsets, lambda U1, U2: U1.issubset(U2)))

print(f'Divisor-closed subsets count: {P_closed.cardinality()}')
print(f'Category placement: {placement_of(P_closed)}')

## 3. Order Isomorphism and Explicit Bijection

In [ ]:
# Verify Sage isomorphism
assert P_div.is_isomorphic(P_closed), 'Posets must be order-isomorphic!'
print('✓ P_div is order-isomorphic to P_closed')

# Define explicit natural morphisms
def phi(d):
    """Forward morphism: Divisor -> Divisor-closed subset of S."""
    return frozenset(s for s in S if d % s == 0)

def psi(U):
    """Inverse morphism: Divisor-closed subset of S -> Divisor."""
    return Integer(lcm(list(U) + [1]))

print('\nExplicit Bijection Table:')
print('=' * 50)
print(f"{'Divisor d':<12} | {'Subset phi(d) in S':<25} | {'psi(phi(d))':<10}")
print('-' * 50)
for d in sorted(divs_120):
    U = phi(d)
    d_rec = psi(U)
    assert d == d_rec, f'Roundtrip failed for {d}'
    subset_str = str(sorted(list(U)))
    print(f"{d:<12} | {subset_str:<25} | {d_rec:<10}")
print('=' * 50)
print('✓ All roundtrips psi(phi(d)) == d verified!')

## 4. Verification of Order Preservation
We verify that $d_1 \mid d_2 \iff \Phi(d_1) \subseteq \Phi(d_2)$ for all $16 \times 16 = 256$ pairs.

In [ ]:
order_preserving = True
for d1 in divs_120:
    for d2 in divs_120:
        div_rel = (d2 % d1 == 0)
        set_rel = phi(d1).issubset(phi(d2))
        if div_rel != set_rel:
            order_preserving = False
            print(f'Mismatch at ({d1}, {d2})')

assert order_preserving, 'Order preservation failed!'
print('✓ phi strictly preserves and reflects the partial order:')
print('  d1 | d2  <===>  phi(d1) subseteq phi(d2)  for all d1, d2 in Div(120)')

## 5. Join-Irreducible Elements & Poset Factorization
The poset of join-irreducible elements of $\mathcal{D}(120)$ is precisely the disjoint sum of chains:
$$
J(\mathcal{D}(120)) = \mathbf{3} \oplus \mathbf{1} \oplus \mathbf{1}
$$
where $\mathbf{3} = \{2 < 4 < 8\}$, $\mathbf{1} = \{3\}$, and $\mathbf{1} = \{5\}$.
Consequently, the divisor lattice factors as a product of chains:
$$
\mathcal{D}(120) \cong \mathcal{O}(\mathbf{3}) \times \mathcal{O}(\mathbf{1}) \times \mathcal{O}(\mathbf{1}) = \mathbf{4} \times \mathbf{2} \times \mathbf{2}
$$

In [ ]:
# Find join-irreducible elements of P_div
# An element is join-irreducible if it has exactly one lower cover in the Hasse diagram
join_irreducibles = [x for x in P_div if len(P_div.lower_covers(x)) == 1]
print(f'Join-irreducibles of P_div: {sorted(join_irreducibles)}')
assert sorted(join_irreducibles) == sorted(S), 'Join-irreducibles must match S = {2, 3, 4, 5, 8}'
print('✓ J(Div(120)) = {2, 3, 4, 5, 8} exactly matches S!')